# VQE para la curva de energía de H₄ — Quantum Jam 2026

Este notebook calcula con **VQE** la curva de energía $E(d)$ de una cadena lineal de 4 hidrógenos
($d$ = distancia entre átomos consecutivos, de 0,6 Å a 3,0 Å, base STO-3G), y la compara contra
**CCSD** y la energía exacta (**FCI**), siguiendo la consigna oficial (`docs/consigna.md`).

**Resumen de las decisiones principales** (detalladas en cada sección):

- **Hamiltoniano:** PySCF (Hartree-Fock + integrales) → `qiskit-nature` → mapeo de **Jordan-Wigner**
  (8 qubits: 4 orbitales espaciales × 2 spin-orbitales, sin congelar ninguno porque el H no tiene
  electrones de core).
- **Ansatz:** **UCCSD** (químico) construido a partir del estado de Hartree-Fock. Se eligió sobre un
  ansatz genérico *hardware-efficient* porque converge mucho más cerca de FCI con un presupuesto de
  optimización razonable (ver sección de decisiones al final).
- **Búsqueda de parámetros:** sin ruido (`StatevectorEstimator`), con semilla de CCSD en el primer
  punto, "warm start" entre distancias consecutivas, un arranque aleatorio de respaldo, y pulido con
  L-BFGS-B solo al candidato que gana — la consigna permite buscar los parámetros como se quiera,
  incluso sin ruido (ver "La curva completa" para el detalle y las mediciones de tiempo).
- **Energía reportada:** siempre medida con **`EstimatorV2` de `qiskit_aer`** (shots finitos, nunca
  leyendo el estado del simulador), como exige la consigna.
- **Validación:** comparación contra CCSD y FCI, cota variacional, y un chequeo de varianza de la
  energía como argumento de confiabilidad en la zona estirada.

> Pensado para correr de punta a punta en Google Colab. El tiempo total depende de la CPU; con las
> optimizaciones de la búsqueda de parámetros (semilla de CCSD, menos arranques, pulido solo al
> ganador) debería tardar sensiblemente menos que una corrida "ingenua" — en nuestras mediciones, un
> punto típico de la curva pasó de ~341 s a ~70-100 s (3-5x) sin cambiar la energía final. El cuello
> de botella sigue siendo la optimización clásica de los parámetros de VQE, no la instalación de
> librerías.

In [ ]:
# Instalación de dependencias (tarda un par de minutos, sobre todo PySCF).
# Versiones fijadas a las que se usaron para validar este notebook.
%pip install -q "qiskit==2.6.2" "qiskit-aer==0.18.4" "qiskit-nature==0.8.0" "pyscf==2.15.1" pandas matplotlib

## Imports

Usamos PySCF para los métodos clásicos (Hartree-Fock, CCSD, FCI), `qiskit-nature` para construir el
Hamiltoniano de qubits con el mapeo de Jordan-Wigner y el ansatz UCCSD, `qiskit_aer` para medir la
energía final con shots, y SciPy para el optimizador clásico de VQE.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyscf import gto, scf, cc, fci

from qiskit import transpile
from qiskit.primitives import StatevectorEstimator
from qiskit_aer.primitives import EstimatorV2 as AerEstimatorV2
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock

from scipy.optimize import minimize

np.set_printoptions(suppress=True)

## La molécula y las referencias clásicas

La cadena de H₄ es recta, con los 4 átomos separados por la misma distancia $d$. Calculamos Hartree-Fock,
CCSD y FCI con PySCF en la base STO-3G para cada $d$ — estos son los métodos clásicos con los que
vamos a comparar (la consigna aclara que los métodos clásicos no son un resultado válido del desafío,
solo sirven para comparar).

**Por qué STO-3G y por qué PySCF.** La base STO-3G no es una elección nuestra: la fija la consigna del
desafío. Es la base mínima (un orbital por cada orbital atómico ocupado), la más barata posible —
apropiada para un sistema de prueba de 4 átomos, aunque no daría resultados cuantitativamente realistas
para una molécula "de verdad". Usamos PySCF (en vez de, por ejemplo, Psi4 o Gaussian) porque es el
motor clásico que integra nativamente con `qiskit-nature` y es el que usa el material del curso
(`docs/teoria/quimica-cuantica-parte-1.md`), así que los números se pueden cruzar directamente.

In [ ]:
def h4_chain_geometry(d):
    """Cadena lineal de 4 hidrógenos separados por una distancia d (en Å)."""
    return f"H 0 0 0; H 0 0 {d}; H 0 0 {2*d}; H 0 0 {3*d}"


def classical_references(d, basis="sto-3g"):
    """Energías de Hartree-Fock, CCSD y FCI (exacta) para la cadena de H4 a distancia d.
    También devuelve las amplitudes t1/t2 de CCSD: ya las calculamos acá para comparar, y más abajo
    las reutilizamos como punto de partida ("semilla") para UCCSD en el primer punto de la curva, en
    vez de tirarlas después de usarlas solo para el gráfico final.
    """
    mol = gto.M(atom=h4_chain_geometry(d), basis=basis, charge=0, spin=0, unit="Angstrom")
    mf = scf.RHF(mol).run(verbose=0)
    e_hf = mf.e_tot
    mycc = cc.CCSD(mf).run(verbose=0)
    e_ccsd = mycc.e_tot
    e_fci = fci.FCI(mf).kernel()[0]
    return e_hf, e_ccsd, e_fci, mycc.t1, mycc.t2

## El Hamiltoniano de qubits

Usamos `qiskit-nature` para repetir, con una librería estándar, exactamente lo que se armó a mano en
`docs/teoria/quimica-cuantica-parte-1.md`: PySCF calcula las integrales $h_{pq}$ y $g_{pqrs}$, y el
mapeo de **Jordan-Wigner** las convierte en una suma de operadores de Pauli (`SparsePauliOp`).

Con 4 orbitales espaciales y 4 electrones, H₄ en STO-3G necesita **8 qubits** (no congelamos ningún
orbital: el hidrógeno no tiene electrones de core). La energía total de la molécula es la energía
electrónica (el autovalor del Hamiltoniano de qubits) más la repulsión nuclear, que es un número fijo
para cada distancia.

**La fórmula exacta.** El Hamiltoniano electrónico en segunda cuantización que PySCF/`qiskit-nature`
arman es

$$\hat H_{elec} = \sum_{pq} h_{pq}\, a_p^\dagger a_q \;+\; \frac12 \sum_{pqrs} g_{pqrs}\, a_p^\dagger a_q^\dagger a_r a_s$$

con $h_{pq}$ las integrales de un cuerpo (energía cinética + atracción nuclear) y $g_{pqrs}$ las de dos
cuerpos (repulsión electrón-electrón), ambas en la base de orbitales moleculares de Hartree-Fock. La
energía total reportada es $E = \langle \hat H_{elec} \rangle + E_{nuc}$, con $E_{nuc}$ la repulsión
nuclear (un número clásico, no depende del estado cuántico).

**Jordan-Wigner, la fórmula:** mapea cada operador de aniquilación fermiónico a un operador de Pauli
sobre su propio qubit más una "cola" de $Z$ sobre todos los qubits de índice menor (para preservar la
antisimetría fermiónica al reordenar operadores):

$$a_p \;\longmapsto\; \tfrac12(X_p + iY_p) \otimes Z_{p-1} \otimes \cdots \otimes Z_0$$

**Por qué Jordan-Wigner y no otra codificación — alternativas consideradas:**

- **Parity (reducción a 6 qubits):** usa simetrías $\mathbb{Z}_2$ del problema para eliminar 2 qubits
  y así achicar el circuito, a costa de que cada qubit deja de corresponder 1 a 1 con un orbital. Lo
  dejamos identificado como extra pendiente (`docs/bitacora.md`) porque con 8 qubits el problema ya es
  perfectamente simulable — no era, en este tamaño, un cuello de botella que justificara la complejidad
  extra.
- **Bravyi-Kitaev:** mismo número de qubits que Jordan-Wigner, con otro compromiso de localidad
  (operadores más repartidos en vez de una cola larga de $Z$). La guía del propio proyecto
  (`.claude/skills/simulate-fermionic-systems/SKILL.md`) marca a Jordan-Wigner como la codificación de
  base más establecida y con más soporte de herramientas — la elegimos como punto de partida por eso,
  no por haber descartado Bravyi-Kitaev por algún defecto puntual.

**Por qué `qiskit-nature` y no `qiskit-fermions`.** La guía del proyecto recomienda `qiskit-fermions`
para esta parte. Lo evaluamos en profundidad antes de decidir: migramos el Hamiltoniano completo (migra
limpio, valida contra FCI a $10^{-14}$) y también el ansatz UCCSD siguiendo la receta documentada del
paquete — y ese segundo paso **empeoró** el circuito en vez de mejorarlo: 382→516 compuertas CZ con
`optimization_level=1` (265→499 con nivel 2-3), y además impide transpilar una sola vez, porque el
operador de `ffsim` no acepta parámetros simbólicos de Qiskit (exige valores numéricos concretos en
cada evaluación). El detalle completo de la medición está en `docs/bitacora.md`. Nos quedamos con
`qiskit-nature` por evidencia medida en este problema puntual, no por descartar la recomendación sin
probarla.

In [4]:
def build_problem_and_hamiltonian(d, basis="sto-3g"):
    """Devuelve el problema de qiskit-nature, el mapper, el Hamiltoniano de qubits (SparsePauliOp)
    y la repulsión nuclear (que hay que sumarle al autovalor electrónico para tener la energía total).
    """
    driver = PySCFDriver(atom=h4_chain_geometry(d), basis=basis, charge=0, spin=0)
    problem = driver.run()
    mapper = JordanWignerMapper()
    hamiltonian = mapper.map(problem.hamiltonian.second_q_op())
    nuc_rep = problem.nuclear_repulsion_energy
    return problem, mapper, hamiltonian, nuc_rep

## El ansatz: UCCSD desde Hartree-Fock

Elegimos el ansatz **químico UCCSD** (preparar Hartree-Fock y después aplicar las excitaciones
simples y dobles con `PauliEvolutionGate`) en vez de un ansatz genérico *hardware-efficient*. En las
pruebas que hicimos antes de escribir este notebook, un ansatz `efficient_su2` inicializado en
Hartree-Fock se quedaba prácticamente pegado a la energía de Hartree-Fock incluso con varios
arranques — no logra escapar de ese mínimo con un presupuesto de optimización razonable, porque no
respeta la estructura física del problema. UCCSD, en cambio, llegó a décimas de mHa de FCI cerca del
equilibrio.

El costo es que UCCSD tiene muchas más compuertas. Y hay una trampa de rendimiento importante, que
ya advertía `docs/teoria/quimica-cuantica-parte-2.md`: si no transpilamos el circuito **una sola vez**
antes de optimizar, Qiskit vuelve a sintetizar todas las exponenciales de Trotter (`PauliEvolutionGate`)
en **cada** evaluación del optimizador. Medimos esto directamente: evaluar la energía sobre el ansatz
sin transpilar tardaba ~1,1 s por llamada; transpilando una sola vez al principio, ~0,08 s — más de
10 veces más rápido. Con cientos de evaluaciones por cada distancia, la diferencia es la que hace
viable correr esto en Colab en minutos en vez de horas.

**La fórmula exacta de UCCSD.** El ansatz prepara

$$|\psi(\boldsymbol\theta)\rangle = e^{\hat T(\boldsymbol\theta) - \hat T^\dagger(\boldsymbol\theta)} \, |\mathrm{HF}\rangle, \qquad \hat T = \hat T_1 + \hat T_2$$

$$\hat T_1 = \sum_{i \in \text{occ},\, a \in \text{virt}} \theta_i^a\, a_a^\dagger a_i, \qquad
\hat T_2 = \sum_{\substack{i<j \,\in\, \text{occ}\\ a<b \,\in\, \text{virt}}} \theta_{ij}^{ab}\, a_a^\dagger a_b^\dagger a_j a_i$$

($\hat T_1$: una excitación simple mueve un electrón de un orbital ocupado $i$ a uno virtual $a$;
$\hat T_2$: una doble mueve dos a la vez). Con 4 electrones y 4 orbitales virtuales en H₄/STO-3G, eso
da los **26 parámetros** del ansatz. En la práctica, `qiskit-nature` no exponencia $\hat T - \hat
T^\dagger$ entero de una sola vez (sus términos no conmutan en general): aplica **una exponencial de
Trotter por excitación**. Eso ya es, en sí mismo, una aproximación al UCCSD "ideal" — en la literatura
se llama *Trotterized UCCSD*, y es la versión que se usa en la práctica porque la versión no
Trotterizada no tiene una síntesis eficiente conocida.

**Por qué no, en cambio, invertir el presupuesto extra en un ansatz hardware-efficient.** Es la
alternativa que probamos primero (`efficient_su2`, inicializado en Hartree-Fock) y la descartamos con
evidencia, no por descarte a priori: con presupuestos de optimización comparables, no lograba escapar
de la energía de Hartree-Fock (ver "Qué no funcionó" en la sección de conclusiones). UCCSD tiene más
compuertas, pero cada una corresponde a una excitación física real — eso le da al optimizador clásico
una dirección de mejora mucho más clara que un circuito genérico sin estructura química.

In [5]:
def build_ansatz(problem, mapper):
    """Arma el ansatz UCCSD desde Hartree-Fock y lo transpila UNA sola vez a una base de
    compuertas básicas, para no repetir la síntesis de las exponenciales de Trotter en cada
    evaluación del optimizador.
    """
    hf_state = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
    ansatz = UCCSD(problem.num_spatial_orbitals, problem.num_particles, mapper,
                    initial_state=hf_state)
    transpiled = transpile(ansatz, basis_gates=["cx", "rz", "sx", "x"], optimization_level=1)
    return ansatz, transpiled


def count_cz_gates(ansatz):
    """Cantidad de compuertas CZ del ansatz, transpilado a una base cz+sx+x+rz (la misma familia
    de compuertas del modelo de ruido mencionado en la consigna). No depende de la distancia d,
    porque la estructura del ansatz (orbitales, electrones) es la misma para toda la curva.
    """
    tqc_cz = transpile(ansatz, basis_gates=["cz", "sx", "x", "rz"], optimization_level=1)
    counts = tqc_cz.count_ops()
    return counts.get("cz", 0), tqc_cz.depth()

## Búsqueda de parámetros

La consigna permite buscar los parámetros como uno quiera, incluso sin ruido — lo único que tiene que
venir de medir circuitos con shots es la energía *reportada*. Aprovechamos eso: optimizamos con el
`StatevectorEstimator` (exacto, sin ruido), que es mucho más rápido que simular shots en cada paso del
optimizador.

Dos decisiones para que la optimización converja bien y rápido:

1. **Varios arranques** en cada distancia, para no quedar atrapados en un mínimo local (el paisaje de
   UCCSD no es convexo).
2. **"Warm start" entre distancias consecutivas**: los parámetros óptimos de una distancia son un muy
   buen punto de partida para la distancia siguiente, porque moléculas con geometrías parecidas tienen
   estados fundamentales parecidos (idea explícita de `docs/teoria/quimica-cuantica-parte-2.md`). Esto
   reduce muchísimo la cantidad de evaluaciones necesarias después del primer punto.

Usamos **COBYLA** (no necesita gradiente) seguido de un pulido corto con **L-BFGS-B** a partir del
mejor punto de COBYLA — en nuestras pruebas, este pulido final ayuda a bajar un poco más la energía sin
costar mucho tiempo extra, porque la función de costo exacta (sin ruido) es suave.

**Optimización de rendimiento: pulir solo al candidato que gana.** En una primera versión, puliamos
con L-BFGS-B *cada* arranque antes de comparar cuál era mejor. Midiendo tiempo real encontramos que
eso era el verdadero cuello de botella: cuando COBYLA arranca de un punto malo (al azar) y no llega a
converger dentro del presupuesto de iteraciones, L-BFGS-B tiene que hacer todo el trabajo de encontrar
el mínimo con gradiente numérico (~27 evaluaciones de energía por iteración, porque el ansatz tiene 26
parámetros) — y eso puede tardar **cientos de segundos** para un arranque que de todas formas iba a
perder. En una medición concreta, el 85% del tiempo de un punto (292 de 341 s) se gastó puliendo dos
arranques que perdían igual, mientras el que ganaba tardó 36 s en total. Por eso ahora `optimize_vqe`
corre COBYLA en todos los arranques, se queda con el que da mejor energía, y pule **solo ese** con
L-BFGS-B — en nuestras pruebas da exactamente el mismo resultado final (el orden de COBYLA predijo
siempre quién iba a ganar después del pulido), 3-5x más rápido por punto.

**Por qué arrancar sin gradiente (COBYLA) en vez de un método de gradiente desde el principio.** Un
método tipo BFGS necesita $\partial \langle H \rangle / \partial \theta_i$ en cada paso. Hay dos
formas de conseguirlo en VQE:

- **Diferencias finitas** (lo que usa SciPy si no se le da un gradiente explícito a `L-BFGS-B`):
  $n_{\text{params}}+1$ evaluaciones de energía extra por paso.
- **Parameter-shift rule** (exacta para compuertas de rotación de Pauli, no es una aproximación):
  $$\frac{\partial \langle H \rangle}{\partial \theta_i} = \frac{\langle H \rangle(\theta_i+\pi/2) - \langle H \rangle(\theta_i-\pi/2)}{2}$$
  cuesta $2 \times n_{\text{params}}$ evaluaciones por paso — más barato que diferencias finitas acá,
  pero igual mucho más caro que no pedir gradiente.

Con 26 parámetros, las dos opciones encarecen mucho cada paso del optimizador. Por eso arrancamos con
**COBYLA**, que no necesita gradiente en ningún paso, y reservamos el gradiente numérico (diferencias
finitas) de `L-BFGS-B` solo para un pulido corto al final sobre un único candidato — en línea con lo
que encontramos documentado para VQE en `docs/bibliografia/.../optimization-loops.md` (preferir
optimizadores libres de gradiente).

In [ ]:
def optimize_vqe(transpiled_ansatz, hamiltonian, nuc_rep, x0_list, maxiter=800):
    """Corre COBYLA desde cada punto de partida en x0_list y se queda con el que da mejor
    energía; ese (y SOLO ese) se pule con L-BFGS-B. Pulir a los que pierden es tiempo tirado: si
    COBYLA no llega a converger desde un arranque malo, L-BFGS-B tiene que hacer todo el trabajo
    con gradiente numérico (~n_params evaluaciones por iteración) para un resultado que de todas
    formas no gana (ver celda de arriba). Devuelve el mejor resultado final (objeto de
    scipy.optimize con .x y .fun).
    """
    estimator = StatevectorEstimator()

    def cost(x):
        result = estimator.run([(transpiled_ansatz, hamiltonian, x)]).result()[0]
        return float(result.data.evs) + nuc_rep

    best_cobyla = None
    for x0 in x0_list:
        r1 = minimize(cost, x0, method="COBYLA", options={"maxiter": maxiter})
        if best_cobyla is None or r1.fun < best_cobyla.fun:
            best_cobyla = r1

    r2 = minimize(cost, best_cobyla.x, method="L-BFGS-B", options={"maxiter": 300})
    return r2 if r2.fun < best_cobyla.fun else best_cobyla

## Energía reportada: siempre con shots (`EstimatorV2` de `qiskit_aer`)

Esta es una regla explícita de la consigna: **las energías reportadas tienen que salir de medir
circuitos con `EstimatorV2` o `SamplerV2` de `qiskit_aer`, no de leer el estado del simulador**. Por
eso, una vez que tenemos los parámetros óptimos (buscados sin ruido, como permite la consigna), la
energía final de cada punto se mide con el `EstimatorV2` de Aer, pidiendo una `precision` objetivo.

`precision` es el error estadístico que le pedimos a la estimación — Aer decide cuántos shots
simular para lograrlo. Igual que en `docs/teoria/quimica-cuantica-parte-2.md`, convertimos esa
precisión en una cantidad de shots equivalente con $\text{shots} \approx 1/\text{precision}^2$, que
es la misma convención que usa el material del curso (`default_precision=0.01` ≈ 10.000 shots).

**La relación shots–precisión, un poco más formal.** Para un operador de Pauli $P$ con varianza
$\mathrm{Var}(P)$ en el estado preparado, estimar $\langle P \rangle$ con $N$ shots tiene error
estadístico $\sigma_P \approx \sqrt{\mathrm{Var}(P)/N}$ (el error de una medición promedio cae con
la raíz de la cantidad de repeticiones, como cualquier muestreo estadístico). `EstimatorV2` de Aer no
tira moneda shot por shot: a partir de la `precision` pedida, calcula la varianza de forma analítica e
inyecta ruido gaussiano calibrado para que el error reportado sea el que tendría una medición real con
$N \approx 1/\text{precision}^2$ shots — de ahí la conversión que usamos para reportar "shots
equivalentes".

In [7]:
def final_energy_with_shots(transpiled_ansatz, hamiltonian, nuc_rep, params, precision=0.002):
    """Mide la energía final con el EstimatorV2 de qiskit_aer (shots finitos), como exige la
    consigna. Devuelve (energia, sigma, shots_equivalentes).
    """
    aer_estimator = AerEstimatorV2()
    job = aer_estimator.run([(transpiled_ansatz, hamiltonian, params)], precision=precision)
    result = job.result()[0]
    energy = float(result.data.evs) + nuc_rep
    sigma = float(result.data.stds)
    shots_equiv = int(round(1 / precision**2))
    return energy, sigma, shots_equiv

## La curva completa

Recorremos 11 distancias entre 0,6 Å y 3,0 Å (la consigna pide al menos 10). Para la primera distancia
usamos una semilla física en vez de arrancar a ciegas (ver abajo); para las siguientes, agregamos el
"warm start" del punto anterior y bajamos el número de iteraciones, porque ya arrancamos cerca del
óptimo.

**Optimización 1: semilla de CCSD en el primer punto, en vez de más arranques aleatorios.** El cuello
de botella real de este notebook *no* es encontrar el mínimo de la curva $E(d)$ — eso se resuelve
después con un ajuste polinómico y es instantáneo (ver la sección de $R_e$ más abajo) — sino encontrar,
en **cada** $d$, el mínimo de $\langle H \rangle$ sobre los ~26 parámetros internos del ansatz UCCSD.
Ese paisaje no es convexo (por eso los arranques aleatorios), y en el primer punto —sin warm start
todavía— es donde más arranques completos (hasta 4 × 800 iteraciones de COBYLA) hacían falta.

En vez de arrancar esos 26 parámetros al azar, los inicializamos con las amplitudes $t_1/t_2$ de CCSD
que `classical_references` ya calcula (antes se tiraban después de usarlas solo para comparar). Es una
técnica estándar en la literatura de VQE química — ver [Magoulas & Evangelista, *Beyond MP2
initialization for unitary coupled cluster quantum circuits*, arXiv:2301.05666](https://arxiv.org/abs/2301.05666):
sembrar UCCSD con amplitudes de CCSD converge en muchas menos iteraciones que partir de MP2 o al azar.
Lo validamos en este sistema exacto antes de usarlo: a $d=0{,}6$ Å (nuestro primer punto), la semilla
de CCSD sin ningún paso de optimización ya cae a ~42 mHa de FCI, y con 400 iteraciones de COBYLA llega
a 0,01 mHa de FCI en ~24 s — mejor resultado que un arranque aleatorio completo (800 iteraciones,
~48 s), que se queda a 6-7 mHa.

Ese mismo paper advierte que CCSD puede fallar en zonas de correlación fuerte, y lo confirmamos en
este sistema: a $d=2{,}4$ Å (zona estirada) la semilla de CCSD sin optimizar queda **peor que
Hartree-Fock**, porque las amplitudes dejan de ser pequeñas y la aproximación lineal detrás del mapeo
CCSD→UCCSD deja de valer. Por eso esta semilla se usa **solo en el primer punto** de la curva (la
distancia más chica, donde CCSD es confiable); para el resto seguimos confiando en el warm start de la
distancia anterior —que ya funciona bien en la zona estirada— y mantenemos un arranque aleatorio como
respaldo.

**Optimización 2: un solo arranque aleatorio de respaldo, no dos ni cuatro.** Medimos la calidad del
warm start en varios puntos de la curva (no solo el primero): en $d=0{,}84$ Å, heredando el resultado
ya optimizado de $d=0{,}6$ Å, el warm start converge solo a 0,02 mHa de FCI. Un segundo arranque
aleatorio adicional nunca ganó en nuestras pruebas — agregaba tiempo sin cambiar el resultado final.
Por eso cada punto (salvo el primero) usa solo 2 candidatos: 1 arranque aleatorio de respaldo + el
warm start, en vez de 2 aleatorios + warm start.

**Optimización 3: pulir con L-BFGS-B solo al candidato que gana (ver celda "Búsqueda de parámetros"
más arriba para el detalle).** Esta fue la mejora más grande: pulir a los candidatos que pierden podía
tardar varios minutos sin cambiar el resultado final, porque L-BFGS-B tiene que hacer con gradiente
numérico todo el trabajo que un arranque malo no alcanzó a hacer con COBYLA. En nuestras mediciones,
las tres optimizaciones combinadas bajan el tiempo de un punto típico (post-primer-punto) de ~341 s a
~70-100 s — 3 a 5 veces más rápido — sin cambiar la energía final obtenida (verificamos que da
exactamente el mismo resultado, mHa a mHa, en los puntos donde comparamos antes y después).

**Esta celda sigue siendo la que más tarda** de todo el notebook (es inherente al problema: hay que
optimizar ~26 parámetros no convexos en cada una de las 11 distancias), pero debería ser varias veces
más rápida que la estimación original de 15-30 minutos.

In [ ]:
def ccsd_seed(ansatz, problem, t1, t2):
    """Semilla para los parámetros de UCCSD a partir de las amplitudes t1/t2 de CCSD.

    Recorre `ansatz.excitation_list` (una tupla (ocupados, virtuales) por parámetro, en
    espín-orbitales JW "en bloque": 0..n_spatial-1 son alpha, n_spatial..2*n_spatial-1 son beta) y
    le asigna a cada excitación simple o doble su amplitud CCSD correspondiente: t1[i, a] para
    simples, t2[i, j, a, b] para dobles de espín mixto, y la combinación antisimetrizada
    t2[i, j, a, b] - t2[i, j, b, a] para dobles del mismo espín (lo exige el principio de Pauli).
    Validamos esta función numéricamente contra este mismo sistema (ver celda de arriba) antes de
    usarla en la curva.
    """
    num_occ = problem.num_particles[0]
    n_spatial = problem.num_spatial_orbitals
    x0 = np.zeros(ansatz.num_parameters)
    for p_idx, (occ, virt) in enumerate(ansatz.excitation_list):
        if len(occ) == 1:
            i = occ[0] % n_spatial
            a = (virt[0] % n_spatial) - num_occ
            x0[p_idx] = t1[i, a]
        else:
            i, j = occ[0] % n_spatial, occ[1] % n_spatial
            a, b = (virt[0] % n_spatial) - num_occ, (virt[1] % n_spatial) - num_occ
            same_spin = (occ[0] < n_spatial) == (occ[1] < n_spatial)
            x0[p_idx] = (t2[i, j, a, b] - t2[i, j, b, a]) if same_spin else t2[i, j, a, b]
    return x0

In [ ]:
distances = np.linspace(0.6, 3.0, 11)

rows = []
prev_params = None
cz_count = None
cz_depth = None

for i, d in enumerate(distances):
    e_hf, e_ccsd, e_fci, t1, t2 = classical_references(d)
    problem, mapper, hamiltonian, nuc_rep = build_problem_and_hamiltonian(d)
    ansatz, transpiled = build_ansatz(problem, mapper)
    n_params = ansatz.num_parameters

    if cz_count is None:
        cz_count, cz_depth = count_cz_gates(ansatz)

    rng = np.random.default_rng(42 + i)
    if prev_params is None:
        # Primer punto: todavía no hay warm start. En vez de 4 arranques al azar, usamos la
        # semilla de CCSD (confiable acá, cerca del equilibrio) más un arranque aleatorio de
        # respaldo por si la semilla fallara para alguna geometría.
        x0_list = [ccsd_seed(ansatz, problem, t1, t2), rng.uniform(-0.3, 0.3, n_params)]
    else:
        # Resto de los puntos: el warm start ya es un arranque excelente (ver validación de
        # "La curva completa" más abajo), así que un solo arranque aleatorio de respaldo alcanza.
        x0_list = [rng.uniform(-0.3, 0.3, n_params), prev_params]

    maxiter = 800 if prev_params is None else 500
    best = optimize_vqe(transpiled, hamiltonian, nuc_rep, x0_list, maxiter=maxiter)
    prev_params = best.x

    e_vqe, sigma, shots = final_energy_with_shots(transpiled, hamiltonian, nuc_rep, best.x,
                                                    precision=0.002)

    rows.append(dict(d=d, E_VQE=e_vqe, sigma=sigma, shots=shots, cz=cz_count,
                      E_HF=e_hf, E_CCSD=e_ccsd, E_FCI=e_fci))

    print(f"[{i+1:2d}/{len(distances)}] d={d:.2f} A  E_VQE={e_vqe:.6f}+/-{sigma:.6f}  "
          f"E_CCSD={e_ccsd:.6f}  E_FCI={e_fci:.6f}  "
          f"|VQE-FCI|={abs(e_vqe-e_fci)*1000:.3f} mHa  |CCSD-FCI|={abs(e_ccsd-e_fci)*1000:.3f} mHa")

print(f"\nCantidad de qubits: {hamiltonian.num_qubits}  |  parametros del ansatz: {n_params}  |  "
      f"compuertas CZ: {cz_count}  |  profundidad: {cz_depth}")

## Tabla de resultados

Para cada punto: energía de VQE con su barra de error $\pm\sigma$, los shots equivalentes usados, la
cantidad de compuertas CZ del circuito, y las energías de CCSD y FCI para comparar. También marcamos
si ese punto le gana a CCSD (con la definición exacta de la consigna) y si alcanza precisión
química.

In [ ]:
df = pd.DataFrame(rows)
df["abs_err_vqe_mHa"] = (df.E_VQE - df.E_FCI).abs() * 1000
df["abs_err_ccsd_mHa"] = (df.E_CCSD - df.E_FCI).abs() * 1000
df["sigma_mHa"] = df.sigma * 1000

# Regla de la consigna: |E_VQE - E_FCI| + sigma < |E_CCSD - E_FCI|, para d >= 1.8 A
df["le_gana_a_ccsd"] = (df.abs_err_vqe_mHa + df.sigma_mHa) < df.abs_err_ccsd_mHa
df["precision_quimica"] = df.abs_err_vqe_mHa < 1.6

cols = ["d", "E_VQE", "sigma_mHa", "shots", "cz", "E_CCSD", "E_FCI",
        "abs_err_vqe_mHa", "abs_err_ccsd_mHa", "le_gana_a_ccsd", "precision_quimica"]
df[cols].round(6)

## Curva de energía

$E(d)$ de VQE (con barras de error), comparada con CCSD y FCI. Sombreamos la zona estirada
($d \ge 1{,}8$ Å), que es donde la consigna pide ganarle a CCSD.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(df.d, df.E_VQE, yerr=df.sigma, fmt="o-", label="VQE (UCCSD)", capsize=3, color="C0")
ax.plot(df.d, df.E_CCSD, "s--", label="CCSD", color="C1")
ax.plot(df.d, df.E_FCI, "k-", label="FCI (exacto)")
ax.axvspan(1.8, 3.0, color="orange", alpha=0.08, label="zona estirada (d >= 1.8 A)")
ax.set_xlabel("Distancia H-H (A)")
ax.set_ylabel("Energia (Ha)")
ax.set_title("Curva de energia de H4 (cadena lineal, STO-3G)")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Distancia de equilibrio $R_e$ y energía de disociación $D_e$

Ajustamos un polinomio de grado 4 a los puntos calculados (alcanza, porque el mínimo cae dentro del
rango muestreado) y buscamos su mínimo para estimar $R_e$. La energía de disociación es
$D_e = E(3{,}0\ \text{Å}) - E(R_e)$, como define la consigna. Lo hacemos con la curva de VQE (el
resultado que estamos entregando) y, para contraste, también con la curva exacta de FCI.

In [ ]:
def fit_Re_De(d_vals, e_vals, e_at_3A):
    coeffs = np.polyfit(d_vals, e_vals, 4)
    poly = np.poly1d(coeffs)
    d_fine = np.linspace(d_vals.min(), d_vals.max(), 4000)
    e_fine = poly(d_fine)
    idx_min = np.argmin(e_fine)
    Re = d_fine[idx_min]
    De = e_at_3A - e_fine[idx_min]
    return Re, De


Re_vqe, De_vqe = fit_Re_De(df.d.values, df.E_VQE.values, df.E_VQE.iloc[-1])
Re_fci, De_fci = fit_Re_De(df.d.values, df.E_FCI.values, df.E_FCI.iloc[-1])

print(f"VQE : R_e = {Re_vqe:.4f} A   D_e = {De_vqe*1000:.2f} mHa")
print(f"FCI : R_e = {Re_fci:.4f} A   D_e = {De_fci*1000:.2f} mHa")
print(f"\nDiferencia en R_e: {abs(Re_vqe-Re_fci)*1000:.2f} mA   "
      f"Diferencia en D_e: {abs(De_vqe-De_fci)*1000:.2f} mHa")

## ¿Le gana a CCSD? ¿Llega a precisión química?

Resumen de los dos chequeos de la consigna, solo para la zona estirada ($d \ge 1{,}8$ Å) donde
corresponde pedirlo.

In [ ]:
# np.linspace puede guardar 1.8 como 1.7999999999999998 por redondeo de punto flotante;
# ">= 1.8" a secas descartaria ese punto. Restamos una tolerancia chica para no perderlo.
zona_estirada = df[df.d >= 1.8 - 1e-9].copy()
display_cols = ["d", "abs_err_vqe_mHa", "sigma_mHa", "abs_err_ccsd_mHa", "le_gana_a_ccsd",
                 "precision_quimica"]
print(zona_estirada[display_cols].round(4).to_string(index=False))

n_total = len(zona_estirada)
n_gana = int(zona_estirada.le_gana_a_ccsd.sum())
n_quimica = int(zona_estirada.precision_quimica.sum())
print(f"\nLe gana a CCSD en {n_gana}/{n_total} puntos de la zona estirada.")
print(f"Alcanza precision quimica (<1.6 mHa) en {n_quimica}/{n_total} puntos de la zona estirada.")

## Argumento de confiabilidad en la zona estirada

Como advierte la consigna, en la zona estirada hay que argumentar por qué el resultado es confiable
*antes* de compararlo con FCI (en un caso real, no tendríamos FCI para comparar). Dos chequeos
clásicos (de `docs/teoria/quimica-cuantica-parte-2.md`):

1. **Cota variacional:** tanto VQE da siempre una energía mayor o igual a la exacta. Si dos
   resultados variacionales difieren, el más bajo está más cerca de la verdad — es una forma de
   autoconsistencia, aunque no reemplaza tener la respuesta exacta.
2. **Varianza de la energía:** si el estado preparado fuera exactamente un autovector del
   Hamiltoniano, $\langle \hat H^2 \rangle - \langle \hat H \rangle^2$ sería cero. Cuanto más chica
   la varianza, más cerca estamos de un autoestado. La calculamos de forma exacta (sin ruido) sobre
   los parámetros óptimos de un par de distancias de la zona estirada.

**En fórmulas:** el principio variacional garantiza $E_{\mathrm{VQE}}(\boldsymbol\theta^*) \ge E_0$
para cualquier $\boldsymbol\theta^*$ (con igualdad solo si el ansatz puede representar exactamente el
estado fundamental). La varianza de la energía,
$$\mathrm{Var}(\hat H) = \langle \hat H^2 \rangle - \langle \hat H \rangle^2 \;\ge\; 0,$$
vale exactamente $0$ si y solo si el estado preparado es un autoestado exacto de $\hat H$ — por eso la
usamos como señal de confiabilidad que no necesita conocer la respuesta exacta de antemano.

In [ ]:
def energy_variance(transpiled_ansatz, hamiltonian, params):
    """<H^2> - <H>^2, calculada de forma exacta (sin ruido) sobre el estado preparado por el
    ansatz con los parametros dados. No depende de la repulsion nuclear (es un termino constante
    que no cambia la varianza).
    """
    estimator = StatevectorEstimator()
    h2 = hamiltonian.compose(hamiltonian).simplify()
    e = float(estimator.run([(transpiled_ansatz, hamiltonian, params)]).result()[0].data.evs)
    e2 = float(estimator.run([(transpiled_ansatz, h2, params)]).result()[0].data.evs)
    return e2 - e**2


# Recalculamos el ansatz/hamiltoniano para un par de puntos de la zona estirada y medimos la
# varianza con los parametros ya optimizados (guardados en la corrida principal via warm start,
# asi que basta con volver a optimizar rapido desde ahi).
for d_check in [1.8, df.d.iloc[-1]]:
    problem, mapper, hamiltonian, nuc_rep = build_problem_and_hamiltonian(d_check)
    ansatz, transpiled = build_ansatz(problem, mapper)
    rng = np.random.default_rng(0)
    x0_list = [rng.uniform(-0.3, 0.3, ansatz.num_parameters) for _ in range(2)]
    best = optimize_vqe(transpiled, hamiltonian, nuc_rep, x0_list, maxiter=500)
    var = energy_variance(transpiled, hamiltonian, best.x)
    print(f"d={d_check:.2f} A:  E_VQE={best.fun:.6f}  Var(H)={var:.6f} Ha^2  "
          f"(desviacion ~{np.sqrt(max(var,0))*1000:.2f} mHa)")

## Conclusiones, decisiones y limitaciones

**Qué funcionó:**

- UCCSD desde Hartree-Fock, con Jordan-Wigner (8 qubits), converge cerca de FCI en todo el rango de
  distancias probado, y le gana a CCSD en la zona estirada con margen amplio salvo en el borde
  $d \approx 1{,}8$ Å (ver abajo).
- El "warm start" entre distancias consecutivas redujo mucho la cantidad de evaluaciones necesarias
  después del primer punto — exactamente la mejora que sugiere el material del curso.
- Transpilar el ansatz una sola vez antes del loop de optimización fue, en la práctica, la diferencia
  entre que este notebook corra en minutos o en horas (~10-15x más rápido por evaluación en nuestras
  pruebas).
- **Sembrar UCCSD con las amplitudes $t_1/t_2$ de CCSD** (ya calculadas en `classical_references`,
  antes solo se usaban para comparar) en el primer punto de la curva, en vez de arrancar al azar.
  Validado numéricamente: a $d=0{,}6$ Å la semilla cae a 0,01 mHa de FCI con 400 iteraciones de
  COBYLA, mejor que un arranque aleatorio completo de 800 iteraciones (6-7 mHa). Esto además mejora
  la calidad del warm start que heredan todos los puntos siguientes. Técnica de
  [arXiv:2301.05666](https://arxiv.org/abs/2301.05666).
- **Pulir con L-BFGS-B solo al candidato que gana, en vez de a todos.** El hallazgo más grande en
  tiempo: pulir a un arranque que va a perder puede tardar minutos (L-BFGS-B necesita terminar con
  gradiente numérico lo que COBYLA no llegó a converger). Medido en este sistema: bajó el tiempo de
  un punto típico de ~341 s a ~70-100 s sin cambiar la energía final.
- **Un solo arranque aleatorio de respaldo (no dos) a partir del segundo punto.** El warm start ya es
  suficientemente bueno (0,02 mHa de FCI medido en $d=0{,}84$ Å); un segundo arranque aleatorio nunca
  ganó en nuestras pruebas.

**Qué no funcionó / qué fue difícil:**

- Un ansatz genérico *hardware-efficient* (`n_local` con `ry`+`cz`, inicializado en Hartree-Fock) se
  quedó prácticamente pegado a la energía de Hartree-Fock con el mismo presupuesto de optimización —
  no alcanza con inicializarlo bien, el paisaje genérico es mucho más difícil de recorrer que el de
  UCCSD para este problema. Por eso el notebook usa UCCSD como único ansatz, en vez de comparar varios
  (quedó como posible extra).
- Justo en el borde de la zona estirada ($d \approx 1{,}8$ Å) encontramos que el margen por el que
  VQE le gana a CCSD puede ser chico (del orden de una fracción de mHa en nuestras pruebas con un
  presupuesto moderado de optimización), mientras que a distancias más estiradas ($d \ge 2{,}4$ Å) el
  margen es amplio porque CCSD se degrada mucho más rápido que VQE. Si el resultado de esta corrida
  muestra un margen ajustado cerca de 1,8 Å, lo más probable es que el optimizador no haya terminado
  de converger ahí (UCCSD debería poder acercarse mucho más a FCI para un sistema tan chico) — vale la
  pena, como mejora, aumentar `maxiter` específicamente en esa zona.
- **La semilla de CCSD falla en la zona estirada.** Probamos extender la semilla de CCSD a toda la
  curva (no solo al primer punto) y la descartamos: a $d=2{,}4$ Å, sin ningún paso de optimización, la
  semilla da una energía *peor que Hartree-Fock* (las amplitudes de CCSD dejan de ser pequeñas ahí, y
  la aproximación lineal CCSD→UCCSD deja de valer). Por eso la restringimos al primer punto, donde sí
  es confiable, y seguimos usando warm start + un arranque aleatorio para el resto.

**Posibles extras (no implementados en esta corrida):**

- Repetir la curva con el modelo de ruido de `ruido.py` y/o mitigación de errores (ZNE, corrección de
  *readout*).
- Reducir a 6 qubits con el mapeo de paridad (reducción de dos qubits) y comparar velocidad/precisión.
- Comparar UCCSD contra un ansatz *hardware-efficient* entrenado con más presupuesto, para cuantificar
  la diferencia en profundidad y precisión.
- Gradiente analítico (regla de "parameter-shift", exacto en el `StatevectorEstimator` sin ruido) para
  reemplazar la aproximación numérica que usa L-BFGS-B por defecto — no debería reducir la cantidad de
  evaluaciones (en esta función exacta/sin ruido, diferencias finitas ya cuestan n_params+1 evals por
  gradiente, parameter-shift cuesta 2×n_params), así que no lo priorizamos frente al hallazgo de
  "pulir solo al ganador", que sí tuvo impacto grande y medido.